# 架空データで jpomop を最初から最後まで動かす

架空の患者 100 人のデータを使い、jpomop でファイルの読み込みから OMOP CDM への変換まで試します。
入力する2つのファイルの中身、仕様との照合、変換後の8つの表を順に見ていきます。

このノートブックに出てくる患者・病院・薬のコード・診療行為のコードは、すべて架空です。
ICD-10 のコードだけは実在のコードですが、実在の患者のものではありません。
このノートブックを動かすのに、病院のデータ、マスター、Athena からのダウンロードは要りません。

jpomop はまだ初期の段階です（バージョン 0.0.1）。
病院の実データでは一度も動かしていません。
このノートブックで確かめられるのは、仕様どおりに作ったファイルを jpomop がどう扱うかです。
実データでどこまで正しく動くかは、まだ分かりません。

次のコマンドで動かせます。

```sh
git clone https://github.com/kojikojikojikoji/jpomop
cd jpomop
pip install -e . pandas jupyter
jupyter lab examples/walkthrough_ja.ipynb
```

英語版は `walkthrough_en.ipynb` です。

## 1. 準備

ファイルはすべて一時フォルダに作ります。
最後のセルでフォルダごと消します。

In [1]:
from pathlib import Path
import tempfile

import duckdb
import pandas as pd

import jpomop

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

work = Path(tempfile.mkdtemp(prefix="jpomop_"))
print("jpomop", jpomop.__version__)

jpomop 0.0.1


## 2. 架空データを作る

`generate` は 2026 年 6 月分のファイルを2つ書き出します。
同じ seed を渡すと、毎回同じ内容のファイルができます。

- `yoshiki1.txt` は様式1 です。入院ごとに、患者・入院・退院・病名が入っています。
- `ef.txt` は入院EF統合ファイルです。算定した項目が、1日1項目につき1行で入っています。

どちらもタブ区切りのテキストで、日本語はシフト JIS です。DPC 調査の実施説明資料が定めている形式に合わせています。jpomop は、Windows のシフト JIS である CP932 として読み込みます。

In [2]:
from jpomop.synth import generate

m = generate(work / "synth", n_patients=100, seed=20261012)
print(m.n_patients, "patients,", m.n_stays, "stays")
for p in (m.yoshiki1, m.ef):
    print(p.name, p.stat().st_size, "bytes,", len(p.read_bytes().splitlines()), "lines")

100 patients, 113 stays
yoshiki1.txt 66443 bytes, 821 lines
ef.txt 313518 bytes, 1707 lines


## 3. 様式1 の中身

様式1 は縦に長いファイルで、1回の入院が何行にも分かれています。
1行は 17 項目です。前の8項目は固定で、後ろの9項目（下の表の payload1〜payload9）の意味は行ごとに変わります。
後ろの9項目に何が入っているかは、`code` の列で決まります。

データ識別番号（`data_id`）が 0000000002 の患者の行を出します。
以降、この患者の変換結果を追います。

In [3]:
from jpomop.spec import load_spec

y1_spec = load_spec("dpc_yoshiki1")
lines = m.yoshiki1.read_bytes().decode("cp932").splitlines()
raw_y1 = pd.DataFrame([ln.split("\t") for ln in lines], columns=y1_spec.field_names())
raw_y1[raw_y1["data_id"] == "0000000002"]

,facility_code,data_id,admission_date,stay_no,summary_no,code,version,seq,payload1,payload2,payload3,payload4,payload5,payload6,payload7,payload8,payload9
6,999999999,0000000002,20260609,0,0,A000010,20140401,0,19521204,1,1000001,,,,,,
7,999999999,0000000002,20260609,0,0,A000020,20140401,0,20260609,1,,,100,0,,,
8,999999999,0000000002,20260609,0,0,A000030,20140401,0,20260617,4,4,0,,,,,
9,999999999,0000000002,20260609,0,0,A006010,20140401,0,,J189,,9000001,8002,,,,肺炎の疑い
10,999999999,0000000002,20260609,0,0,A006020,20140401,0,,J189,,9000001,8002,,,,肺炎の疑い
11,999999999,0000000002,20260609,0,0,A006030,20140401,0,,J189,,9000001,8002,,,,肺炎の疑い
12,999999999,0000000002,20260609,0,0,A006040,20140401,1,,E785,,9000103,,,,,脂質異常症


各行は次のように読みます。

| `code` | 行の内容 | 上の表で使っている後ろの9項目 |
|---|---|---|
| A000010 | 患者 | 1: 生年月日、2: 性別、3: 郵便番号 |
| A000020 | 入院 | 1: 入院年月日、2: 入院経路 |
| A000030 | 退院 | 1: 退院年月日、2: 退院先、3: 退院時転帰、4: 24時間以内の死亡 |
| A006010 | 主傷病 | 2: ICD-10、4: 傷病名コード、5: 修飾語コード、9: 傷病名 |
| A006020 | 入院の契機となった傷病 | A006010 と同じ |
| A006030 | 医療資源を最も投入した傷病 | A006010 と同じ |
| A006040 | 入院時併存症 | A006010 と同じ |

この患者では、同じ病名 J189（肺炎）が3行に入っています。
3行とも修飾語コード 8002 が付いています。
実施説明資料では、8002 は「の疑い」を表します。
jpomop がこの病名をどう扱うかは 8 章で見ます。

`summary_no` は、実施説明資料の「統括診療情報番号」です。
0 は、入院全体を1枚にまとめた様式1 を表します。
0 以外の例は 10 章で見ます。

jpomop は、ファイルの形式を YAML ファイルで定義しています。
項目ごとに、位置・型・桁数・実施説明資料のページ番号を書いてあります。

In [4]:
pd.DataFrame(
    [(f.col, f.name, f.name_ja, f.type, f.length, f.page) for f in y1_spec.fields],
    columns=["col", "name", "name_ja", "type", "length", "manual_page"],
).astype({"length": "Int64"})

,col,name,name_ja,type,length,manual_page
0,1,facility_code,施設コード,digits,9,22
1,2,data_id,データ識別番号,digits,10,22
2,3,admission_date,入院年月日,date8,8,22
3,4,stay_no,回数管理番号,string,<NA>,22
4,5,summary_no,統括診療情報番号,string,<NA>,22
5,6,code,コード,string,7,22
6,7,version,バージョン,date8,8,22
7,8,seq,連番,integer,<NA>,22
8,9,payload1,ペイロード1,string,<NA>,22
9,10,payload2,ペイロード2,string,<NA>,22


## 4. EF ファイルの中身

EF ファイルは1行が 31 項目です。
1行に、ある日に算定した1つの項目が入ります。薬、検査、手術などの診療行為、毎日の入院基本料が、それぞれ1行になります。
同じ患者について、31 項目のうち9項目を出します。

In [5]:
ef_spec = load_spec("dpc_ef")
lines = m.ef.read_bytes().decode("cp932").splitlines()
raw_ef = pd.DataFrame([ln.split("\t") for ln in lines], columns=ef_spec.field_names())
cols = ["data_id", "admission_date", "exec_date", "data_kubun", "receipt_code",
        "item_name", "quantity", "detail_flags", "ward_code"]
raw_ef.loc[raw_ef["data_id"] == "0000000002", cols]

,data_id,admission_date,exec_date,data_kubun,receipt_code,item_name,quantity,detail_flags,ward_code
6,0000000002,20260609,20260609,90,190999901,架空入院基本料,0000000.000,000000000000,W1
7,0000000002,20260609,20260609,60,160999901,架空検査（血液）,0000000.000,000000000000,W1
8,0000000002,20260609,20260610,90,190999901,架空入院基本料,0000000.000,000000000000,W1
9,0000000002,20260609,20260610,33,640999903,架空注射液Ｃ ５００ｍＬ,0000001.000,000000000000,W1
10,0000000002,20260609,20260610,70,170999901,架空画像診断,0000000.000,000000000000,W1
11,0000000002,20260609,20260611,90,190999901,架空入院基本料,0000000.000,000000000000,W1
12,0000000002,20260609,20260611,33,640999903,架空注射液Ｃ ５００ｍＬ,0000001.000,000000000000,W1
13,0000000002,20260609,20260611,21,620999901,架空錠Ａ ５ｍｇ,0000001.000,000000000000,W1
14,0000000002,20260609,20260612,90,190999901,架空入院基本料,0000000.000,000000000000,W1
15,0000000002,20260609,20260612,21,620999901,架空錠Ａ ５ｍｇ,0000001.000,000000000000,W1


各行は次のように読みます。

- `exec_date` は実施年月日です。入院は 6 月 9 日から 17 日までなので、9 日分の行があります。
- `data_kubun` はデータ区分で、項目の種類を表します。21 は内服、33 は注射、60 は検査、70 は画像診断、90 は入院基本料です。
- `receipt_code` はレセプト電算処理システム用コードです。実際のファイルでは基本マスターのコードが入ります。ここでは架空のコードを入れています。医薬品マスターのコードは 6 で、医科診療行為マスターのコードは 1 で始まります。架空のコードもこれに合わせています。
- `ward_code` はその日の病棟コードです。この患者は 6 月 14 日に W1 から W2 へ移っています。病棟コードは病院ごとに決めるコードです。
- `detail_flags` は行為明細区分情報で、12 桁あります。3 桁目が持参薬の印です。ここではすべて 0 です。

ファイル全体の行数を、項目の種類ごとに数えます。

In [6]:
from jpomop.spec import load_codes

kubun = load_codes("ef_data_kubun")
t = raw_ef.groupby(["data_kubun", "receipt_code", "item_name"]).size().rename("ef_lines").reset_index()
t.insert(1, "data_kubun_name", t["data_kubun"].map(kubun))
t

,data_kubun,data_kubun_name,receipt_code,item_name,ef_lines
0,21,内服,620999901,架空錠Ａ ５ｍｇ,243
1,21,内服,620999902,架空カプセルＢ,208
2,33,その他（注射）,640999903,架空注射液Ｃ ５００ｍＬ,241
3,50,手術,150999901,架空手術,34
4,60,検査・病理,160999901,架空検査（血液）,141
5,70,画像診断,170999901,架空画像診断,150
6,90,入院基本料,190999901,架空入院基本料,690


## 5. 読み込んで仕様と照合する

`read_yoshiki1` と `read_ef` は、ファイルを読み込む際に、すべての項目を仕様と照合します。
`pivot_yoshiki1` は、縦に長い様式1 を、様式1 ごとに1行の表と、病名ごとに1行の表に組み直します。

架空データは仕様どおりに作ってあるので、違反は 0 件です。

In [7]:
from jpomop.parse.ef import read_ef
from jpomop.parse.yoshiki1 import pivot_yoshiki1, read_yoshiki1

y1 = read_yoshiki1(m.yoshiki1)
tables = pivot_yoshiki1(y1.table)
ef = read_ef(m.ef)

print("Form 1 lines:", y1.table.num_rows)
print("forms:", tables.forms.num_rows, " diagnoses:", tables.diagnoses.num_rows)
print("EF lines:", ef.table.num_rows)
print("violations:", len(y1.violations) + len(tables.violations) + len(ef.violations))

Form 1 lines: 821
forms: 113  diagnoses: 448
EF lines: 1707
violations: 0


組み直した後の、この患者の様式1 です。1枚だけあります。
郵便番号は残していません。

In [8]:
forms = tables.forms.to_pandas()
forms[forms["data_id"] == "0000000002"].T

,1
facility_code,999999999
data_id,0000000002
admission_date,2026-06-09
stay_no,0
summary_no,0
birth_date,1952-12-04
sex,1
admission_route,1
admission_type,100
ambulance,0


性別・入院経路・退院先・退院時転帰のコードは、実施説明資料に定義があります。
jpomop はこれを小さな表として持っています。
この患者は男性（1）で、家庭から入院し（1）、他の病院へ転院しました（4）。転帰は「不変」（4）です。

In [9]:
for name in ["y1_sex", "y1_admission_route", "y1_discharge_to", "y1_discharge_outcome"]:
    print(name, load_codes(name))

y1_sex {'1': '男', '2': '女'}
y1_admission_route {'0': '院内の他病棟からの転棟', '1': '家庭からの入院', '4': '他の病院・診療所の病棟からの転院', '5': '介護施設・福祉施設に入所中', '8': '院内で出生', '9': 'その他'}
y1_discharge_to {'0': '院内の他病棟への転棟', '1': '家庭への退院（当院に通院）', '2': '家庭への退院（他の病院・診療所に通院）', '3': '家庭への退院（その他）', '4': '他の病院・診療所への転院', '5': '介護老人保健施設に入所', '6': '介護老人福祉施設に入所', '7': '社会福祉施設、有料老人ホーム等に入所', '8': '終了（死亡等）', '9': 'その他', 'a': '介護医療院', 'b': '未転棟、未退院'}
y1_discharge_outcome {'1': '治癒・軽快', '3': '寛解', '4': '不変', '5': '増悪', '6': '最も医療資源を投入した傷病による死亡', '7': '最も医療資源を投入した傷病以外による死亡', '9': 'その他'}


同じ様式1 の病名です。
`role` は、その病名が様式1 のどの行から来たかを表します。
`suspected` は、修飾語コード 8002 が付いていれば True になります。

In [10]:
dx = tables.diagnoses.to_pandas()
dx.loc[dx["data_id"] == "0000000002",
       ["role", "seq", "icd10", "disease_code", "modifiers", "suspected", "disease_name"]]

,role,seq,icd10,disease_code,modifiers,suspected,disease_name
3,dx_main,0,J189,9000001,8002,True,肺炎の疑い
4,dx_trigger,0,J189,9000001,8002,True,肺炎の疑い
5,dx_resource1,0,J189,9000001,8002,True,肺炎の疑い
6,dx_comorbid,1,E785,9000103,NaN,False,脂質異常症


## 6. 壊れたファイルを読ませる

照合の動きを見るため、次のセルで誤りを4つ入れたファイルを作ります。

1. 生年月日を 2 月 30 日にする
2. 性別のコードを、定義に無い 9 にする
3. 様式1 の1行を 16 項目にする
4. EF の実施年月日を、退院日より後にする

In [11]:
bad_lines = m.yoshiki1.read_bytes().decode("cp932").splitlines()

a = bad_lines[0].split("\t")   # first line: birth date and sex
a[8] = "19520230"               # 30 February
a[9] = "9"                      # sex code that is not in the manual
bad_lines[0] = "\t".join(a)

b = bad_lines[1].split("\t")   # second line: drop the last field
bad_lines[1] = "\t".join(b[:-1])

bad_y1 = work / "yoshiki1_bad.txt"
bad_y1.write_bytes(("\r\n".join(bad_lines) + "\r\n").encode("cp932"))

bad_ef_lines = m.ef.read_bytes().decode("cp932").splitlines()
c = bad_ef_lines[0].split("\t")
c[23] = "20260701"              # service date after discharge
bad_ef_lines[0] = "\t".join(c)
bad_ef = work / "ef_bad.txt"
_ = bad_ef.write_bytes(("\r\n".join(bad_ef_lines) + "\r\n").encode("cp932"))

誤り1つにつき1行が出ます。
ファイル名、行番号、項目、値、違反した規則が分かります。

In [12]:
from jpomop.parse.cross import check_cross_file

r1 = read_yoshiki1(bad_y1)
t1 = pivot_yoshiki1(r1.table)
r2 = read_ef(bad_ef)
found = r1.violations + t1.violations + r2.violations
found += check_cross_file(r2.table, t1.forms, ef_file=bad_ef.name, y1_file=bad_y1.name)

pd.DataFrame([vars(v) for v in found])

,file,line,field,value,rule,message
0,yoshiki1_bad.txt,2,*,16,n_fields,expected 17 fields
1,yoshiki1_bad.txt,1,A000010.1:birth_date,19520230,date_invalid,not a calendar date
2,yoshiki1_bad.txt,1,A000010.2:sex,9,code_not_in_table,not in the code table
3,ef_bad.txt,1,exec_date,2026-07-01,exec_outside_stay,date of service is outside the stay


コマンドでも同じ照合ができます。

```sh
jpomop validate --yoshiki1 yoshiki1.txt --ef ef.txt --out violations.csv
```

違反が1件でもあると、コマンドは終了コード 1 で終わります。
照合の対象は、jpomop が読み込む項目だけです。
提出用の形式チェックの代わりにはなりません。

## 7. OMOP CDM に変換する

`run_etl` は2つのファイルを読み込み、CDM の表を1つの DuckDB ファイルに書き出します。
コマンドでは `jpomop convert` が同じことをします。

処理結果の報告には、表ごとの行数が入っています。
変換しなかったものと、その理由ごとの件数も入っています。

In [13]:
from jpomop.etl import run_etl

cdm_path = work / "cdm.duckdb"
report = run_etl([m.yoshiki1], [m.ef], cdm_path)

print(pd.Series(report.rows, name="rows").to_string())
print()
print("forms not used:", report.skipped_forms)
print("rows left out:", report.skipped_rows)
print("violations:", report.n_violations, " consistency problems:", report.invariant_problems)

person                  100
observation_period      100
visit_occurrence        113
visit_detail            138
death                     5
condition_occurrence    213
drug_exposure           692
procedure_occurrence    325

forms not used: {}
rows left out: {'condition_occurrence': {'duplicate_in_stay': 208, 'suspected': 27}, 'drug_exposure': {}}
violations: 0  consistency problems: []


報告の各項目を見ます。

- 患者は 100 人、入院は 113 回です。`person` が 100 行、`visit_occurrence` が 113 行になります。
- `duplicate_in_stay` は 208 件です。先ほど見た患者では、様式1 の同じ病名が3行に入っています。jpomop は、1回の入院の中で同じ ICD-10 を1行にまとめます。
- `suspected` は 27 件です。修飾語コード 8002 が付いた病名の行で、`condition_occurrence` には入れていません。
- 「consistency problems」は、基本的な決まりを破った行の一覧です。たとえば、観察期間の外にある記録がこれに当たります。ここでは空です。

次の章から、この患者について表を1つずつ見ます。

## 8. 1人の患者を表ごとに追いかける

### person

患者1人につき1行です。
`person_source_value` には、施設コードとデータ識別番号から計算したハッシュ値を入れています。元の番号は出力に残りません。
`gender_concept_id` の 8507 は、OMOP で男性を表す番号です。
DPC には人種や民族の項目が無いので、その番号は 0 にしています。

In [14]:
con = duckdb.connect(str(cdm_path), read_only=True)

con.sql("""
    SELECT person_id, person_source_value, gender_source_value, gender_concept_id,
           year_of_birth, month_of_birth, day_of_birth
    FROM person WHERE person_id = 2
""").df()

,person_id,person_source_value,gender_source_value,gender_concept_id,year_of_birth,month_of_birth,day_of_birth
0,2,dcc269630e6028ae,1,8507,1952,12,4


### visit_occurrence

入院1回につき1行です。
`visit_concept_id` の 9201 は入院（Inpatient Visit）を表します。
`visit_type_concept_id` の 32810 は、請求データ（Claim）から作った記録であることを表します。
source value の 1 と 4 は、DPC のコードのままです。「家庭からの入院」と「他の病院・診療所への転院」です。

In [15]:
con.sql("""
    SELECT visit_occurrence_id, visit_concept_id, visit_start_date, visit_end_date,
           visit_type_concept_id, admitted_from_source_value, discharged_to_source_value
    FROM visit_occurrence WHERE person_id = 2
""").df()

,visit_occurrence_id,visit_concept_id,visit_start_date,visit_end_date,visit_type_concept_id,admitted_from_source_value,discharged_to_source_value
0,2,9201,2026-06-09,2026-06-17,32810,1,4


### visit_detail

同じ病棟にいた期間ごとに1行です。
この患者は、W1 に 5 日、続けて W2 に 4 日いました。
2 行目の `preceding_visit_detail_id` は、1 行目を指しています。

In [16]:
con.sql("""
    SELECT visit_detail_id, visit_occurrence_id, visit_detail_start_date,
           visit_detail_end_date, visit_detail_source_value, preceding_visit_detail_id
    FROM visit_detail WHERE person_id = 2 ORDER BY visit_detail_start_date
""").df()

,visit_detail_id,visit_occurrence_id,visit_detail_start_date,visit_detail_end_date,visit_detail_source_value,preceding_visit_detail_id
0,2,2,2026-06-09,2026-06-13,W1,<NA>
1,3,2,2026-06-14,2026-06-17,W2,2


病棟は、様式1 ではなく EF ファイルから作っています。
EF ファイルで、この入院の日ごとの病棟コードを並べると次のようになります。
上の2行と一致します。

In [17]:
(raw_ef[raw_ef["data_id"] == "0000000002"]
 .groupby("exec_date")["ward_code"].agg(lambda s: ",".join(sorted(set(s))))
 .to_frame().T)

exec_date,20260609,20260610,20260611,20260612,20260613,20260614,20260615,20260616,20260617
ward_code,W1,W1,W1,W1,W1,W2,W2,W2,W2


### condition_occurrence

この患者の様式1 には、病名の行が4つありました。
そのうち3つは、疑いの付いた肺炎です。
表に入っているのは、残りの1つ（E785）だけです。

In [18]:
con.sql("""
    SELECT condition_occurrence_id, condition_start_date, condition_source_value,
           condition_concept_id, condition_status_source_value, condition_status_concept_id
    FROM condition_occurrence WHERE person_id = 2
""").df()

,condition_occurrence_id,condition_start_date,condition_source_value,condition_concept_id,condition_status_source_value,condition_status_concept_id
0,2,2026-06-09,E785,0,dx_comorbid,32907


様式1 には、病名ごとの日付がありません。
そのため `condition_start_date` には入院年月日を入れています。
病気が始まった日を示すものではありません。

`condition_status_concept_id` は、その病名が様式1 のどの行から来たかを表します。
32907 は「Secondary admission diagnosis」で、jpomop は入院時併存症にこの番号を当てています。
どの番号を当てるかは私の解釈で、公式に決まった対応ではありません。

比べるために、データ識別番号 0000000001 の患者も見ます。
I639 が3行に入っていて、疑いは付いていません。
表では1行になり、区分は主傷病が残ります。

In [19]:
print(dx.loc[dx["data_id"] == "0000000001", ["role", "icd10", "suspected"]].to_string(index=False))
con.sql("""
    SELECT condition_start_date, condition_source_value, condition_status_source_value
    FROM condition_occurrence WHERE person_id = 1
""").df()

        role icd10  suspected
     dx_main  I639      False
  dx_trigger  I639      False
dx_resource1  I639      False


,condition_start_date,condition_source_value,condition_status_source_value
0,2026-06-01,I639,dx_main


### drug_exposure

EF ファイルの薬の行1つにつき1行です。
開始日は EF ファイルの実施年月日です。
`quantity` は EF ファイルの使用量です。
`route_source_value` にはデータ区分（21 内服、33 注射）を、`dose_unit_source_value` には単位コードを残しています。

語彙を渡していないので、`drug_concept_id` は 0 です。
語彙を渡した場合の変換は 11 章で見ます。

In [20]:
con.sql("""
    SELECT drug_exposure_id, drug_exposure_start_date, drug_source_value, drug_concept_id,
           quantity, route_source_value, dose_unit_source_value, visit_occurrence_id
    FROM drug_exposure WHERE person_id = 2 ORDER BY drug_exposure_start_date, drug_exposure_id
""").df()

,drug_exposure_id,drug_exposure_start_date,drug_source_value,drug_concept_id,quantity,route_source_value,dose_unit_source_value,visit_occurrence_id
0,4,2026-06-10,640999903,0,1.0,33,033,2
1,5,2026-06-11,640999903,0,1.0,33,033,2
2,6,2026-06-11,620999901,0,1.0,21,009,2
3,7,2026-06-12,620999901,0,1.0,21,009,2
4,8,2026-06-12,620999902,0,2.0,21,009,2
5,9,2026-06-14,620999902,0,2.0,21,009,2
6,10,2026-06-14,620999901,0,1.0,21,009,2
7,11,2026-06-16,620999901,0,1.0,21,009,2
8,12,2026-06-16,620999902,0,2.0,21,009,2


### procedure_occurrence

EF ファイルのうち、コードが 1 で始まる行1つにつき1行です。入院基本料と食事は除いています。
この患者では、血液検査が 2 行、画像診断が 3 行です。
入院基本料の 9 行は入っていません。

`procedure_concept_id` は、今の版ではすべて 0 です。
日本の診療行為コードを標準の番号に対応づける処理は、まだありません。

In [21]:
con.sql("""
    SELECT procedure_occurrence_id, procedure_date, procedure_source_value,
           procedure_concept_id, visit_occurrence_id
    FROM procedure_occurrence WHERE person_id = 2 ORDER BY procedure_date
""").df()

,procedure_occurrence_id,procedure_date,procedure_source_value,procedure_concept_id,visit_occurrence_id
0,2,2026-06-09,160999901,0,2
1,3,2026-06-10,170999901,0,2
2,4,2026-06-14,160999901,0,2
3,5,2026-06-15,170999901,0,2
4,6,2026-06-16,170999901,0,2


### observation_period

患者1人につき1行です。ファイルに入っている最初の入院日から、最後の退院日までを期間にしています。
person_id が 5 の患者は入院が2回あるので、期間は2回の入院をまたいでいます。

DPC のファイルから分かるのは、入院していた間のことだけです。
入院と入院の間は、実際には何も記録されていません。
観察期間の長さを使う研究では、この点に注意が要ります。

In [22]:
con.sql("""
    SELECT o.person_id, o.observation_period_start_date, o.observation_period_end_date,
           v.visit_occurrence_id, v.visit_start_date, v.visit_end_date
    FROM observation_period o JOIN visit_occurrence v USING (person_id)
    WHERE o.person_id = 5 ORDER BY v.visit_start_date
""").df()

,person_id,observation_period_start_date,observation_period_end_date,visit_occurrence_id,visit_start_date,visit_end_date
0,5,2026-06-06,2026-06-21,5,2026-06-06,2026-06-12
1,5,2026-06-06,2026-06-21,6,2026-06-18,2026-06-21


## 9. 死亡

DPC のデータに死亡が残るのは、入院中に亡くなった場合だけです。
jpomop は、様式1 の退院時転帰が 6 か 7 のときに `death` の行を作ります。
死亡日は退院年月日です。

In [23]:
con.sql("""
    SELECT d.person_id, d.death_date, d.death_type_concept_id,
           v.visit_start_date, v.visit_end_date, v.discharged_to_source_value
    FROM death d JOIN visit_occurrence v
      ON v.person_id = d.person_id AND v.visit_end_date = d.death_date
    ORDER BY d.person_id
""").df()

,person_id,death_date,death_type_concept_id,visit_start_date,visit_end_date,discharged_to_source_value
0,4,2026-06-05,32810,2026-06-02,2026-06-05,8
1,13,2026-06-09,32810,2026-06-05,2026-06-09,8
2,21,2026-06-10,32810,2026-06-06,2026-06-10,8
3,25,2026-06-21,32810,2026-06-20,2026-06-21,8
4,98,2026-06-08,32810,2026-06-02,2026-06-08,8


5 件の死亡の元になった様式1 です。
転帰 6 は、医療資源を最も投入した傷病による死亡です。
転帰 7 は、それ以外の原因による死亡です。
退院した後の死亡は DPC のデータに無いので、`death` の表にも入りません。

In [24]:
forms.loc[forms["discharge_outcome"].isin(["6", "7"]),
          ["data_id", "admission_date", "discharge_date", "discharge_outcome"]]

,data_id,admission_date,discharge_date,discharge_outcome
3,0000000004,2026-06-02,2026-06-05,7
15,0000000013,2026-06-05,2026-06-09,7
23,0000000021,2026-06-06,2026-06-10,6
28,0000000025,2026-06-20,2026-06-21,7
110,0000000098,2026-06-02,2026-06-08,7


## 10. 数えるだけで変換しない様式1

病院は、1回の入院について様式1 を複数作ることがあります。
たとえば、患者が病棟グループをまたいで移ると、入院全体の様式1 に加えて、病棟グループごとの様式1 を作ります。
入院全体の様式1 は `summary_no`（統括診療情報番号）が 0 です。
病棟グループごとの様式1 は 1、2 と続きます。

架空データには、入院全体の様式1 しかありません。
そこで次のセルでは、データ識別番号 0000000001 の患者の様式1 を写し、`summary_no` を 1 に変えてファイルに足します。
病棟グループごとの様式1 を、大まかにまねたものです。

In [25]:
src = m.yoshiki1.read_bytes().decode("cp932").splitlines()
extra = []
for ln in src:
    f = ln.split("\t")
    if f[1] == "0000000001":
        f[4] = "1"              # 統括診療情報番号: 0 -> 1 (ward-group form)
        extra.append("\t".join(f))

y1_more = work / "yoshiki1_with_ward_group_form.txt"
y1_more.write_bytes(("\r\n".join(src + extra) + "\r\n").encode("cp932"))

report2 = run_etl([y1_more], [m.ef], work / "cdm2.duckdb")
print("forms not used:", report2.skipped_forms)
print("visit_occurrence rows:", report2.rows["visit_occurrence"],
      " condition_occurrence rows:", report2.rows["condition_occurrence"])

forms not used: {'summary_no=1': 1}
visit_occurrence rows: 113  condition_occurrence rows: 213


足した様式1 は「forms not used」に数えられています。
入院の数と病名の行数は前と同じ（113 と 213）なので、入院を二重に数えていません。

7 日以内の再入院をまとめた様式1（A と B）も、同じ扱いにしています。
入院 90 日目の仮の様式1（Z）は、その入院にまだ入院全体の様式1 が無いときだけ使います。
この扱いが正しいかどうかは、実データではまだ確かめていません。

## 11. concept ID と語彙

ここまでの変換では、病名・薬・診療行為の concept ID はすべて 0 です。
元のコードは source value の列に残っています。

In [26]:
con.sql("""
    SELECT 'condition_occurrence' AS cdm_table, count(*) AS n_rows,
           count(*) FILTER (WHERE condition_concept_id = 0) AS concept_id_0
    FROM condition_occurrence
    UNION ALL SELECT 'drug_exposure', count(*),
           count(*) FILTER (WHERE drug_concept_id = 0) FROM drug_exposure
    UNION ALL SELECT 'procedure_occurrence', count(*),
           count(*) FILTER (WHERE procedure_concept_id = 0) FROM procedure_occurrence
""").df()

,cdm_table,n_rows,concept_id_0
0,condition_occurrence,213,213
1,drug_exposure,692,692
2,procedure_occurrence,325,325


concept ID を入れるには、次の3つのファイルが要ります。どれも、使う人が自分でダウンロードします。

- Athena の語彙（ICD10、SNOMED、RxNorm、RxNorm Extension）
- 医薬品マスター（薬のレセプト電算処理システム用コードと、薬価基準収載医薬品コードの対応が入っています）
- 木村らの対応表（2025 年。薬価基準収載医薬品コードと OMOP の concept の対応が入っています）

このノートブックには、これらのファイルを入れていません。
次のセルでは代わりに、同じ形をした小さなファイルを3つ作ります。
中の concept ID はすべて架空で、9000 で始まる番号にしています。
Athena には存在しない番号です。

対応が見つからないコードの動きも見るため、ICD-10 は 11 種類のうち 8 種類だけ、薬は 3 種類のうち 2 種類だけを入れています。

In [27]:
athena = work / "athena_invented"
athena.mkdir()

# (ICD-10 code, invented ICD10 concept ID, invented standard concept ID)
icd = [("J18.9", 900000001, 900000101), ("I63.9", 900000002, 900000102),
       ("I50.0", 900000003, 900000103), ("K80.2", 900000004, 900000104),
       ("S72.0", 900000005, 900000105), ("C18.9", 900000006, 900000106),
       ("E11.9", 900000007, 900000107), ("I10", 900000008, 900000108)]

concept = ["concept_id\tconcept_name\tdomain_id\tvocabulary_id\tconcept_class_id\t"
           "standard_concept\tconcept_code\tvalid_start_date\tvalid_end_date\tinvalid_reason"]
relationship = ["concept_id_1\tconcept_id_2\trelationship_id\tvalid_start_date\t"
                "valid_end_date\tinvalid_reason"]
for code, source_id, standard_id in icd:
    concept.append(f"{source_id}\tinvented {code}\tCondition\tICD10\tICD10 code\t\t{code}"
                   "\t19700101\t20991231\t")
    concept.append(f"{standard_id}\tinvented standard for {code}\tCondition\tSNOMED\t"
                   f"Clinical Finding\tS\tX{standard_id}\t19700101\t20991231\t")
    relationship.append(f"{source_id}\t{standard_id}\tMaps to\t19700101\t20991231\t")
(athena / "CONCEPT.csv").write_text("\n".join(concept) + "\n", encoding="utf-8")
(athena / "CONCEPT_RELATIONSHIP.csv").write_text("\n".join(relationship) + "\n", encoding="utf-8")

# A file shaped like the drug master: 42 fields, receipt code in field 3,
# NHI drug price code in field 32. Both codes are invented.
pairs = {"620999901": "9999001F1011", "620999902": "9999002M1011", "640999903": "9999400A1011"}
rows = []
for receipt, price_code in pairs.items():
    fields = ["0", "Y", receipt] + [""] * 39
    fields[31] = price_code
    rows.append(",".join(f'"{x}"' for x in fields))
drug_master = work / "y_invented.csv"
drug_master.write_bytes(("\r\n".join(rows) + "\r\n").encode("cp932"))

# A file shaped like rxnorm_mapping.tsv. The third drug has no match on purpose.
mapping = work / "rxnorm_mapping_invented.tsv"
_ = mapping.write_text("Category\tNHIDPLC \tOSV_ID\tOSV_NAME\n"
                   "PO\t9999001F1011\t900001001\tinvented drug A\n"
                   "PO\t9999002M1011\t900001002\tinvented drug B\n"
                   "Injection\t9999400A1011\tN/A\tN/A\n", encoding="utf-8")

`build_vocab` は、3つのファイルを1つの小さな DuckDB ファイルにまとめます。

DPC の ICD-10 にはピリオドが無く、5 桁目が付くこともあります。
jpomop は、細かいコードから順に対応を探します。
S7200 は、架空の語彙に S72.00 が無いので、S72.0 に当たります。
N390 は対応が無いので、concept ID は 0 のままです。

In [28]:
from jpomop.vocab import Vocab, build_vocab, icd10_candidates

vocab_path = work / "vocab.duckdb"
vr = build_vocab(athena, [drug_master], mapping, vocab_path)
print(vr)

v = Vocab.load(vocab_path)
for code in ["J189", "S7200", "N390"]:
    print(code, icd10_candidates(code), "->", v.map_icd10(code))
for code in ["620999901", "640999903"]:
    print(code, "->", v.map_drug(code))

VocabBuildReport(icd10_codes=8, icd10_mapped=8, receipt_drugs=3, receipt_drugs_mapped=2)
J189 ['J18.9', 'J18'] -> ConceptMatch(concept_id=900000101, source_concept_id=900000001, source_value='J189')
S7200 ['S72.00', 'S72.0', 'S72'] -> ConceptMatch(concept_id=900000105, source_concept_id=900000005, source_value='S7200')
N390 ['N39.0', 'N39'] -> ConceptMatch(concept_id=0, source_concept_id=0, source_value='N390')
620999901 -> ConceptMatch(concept_id=900001001, source_concept_id=0, source_value='620999901')
640999903 -> ConceptMatch(concept_id=0, source_concept_id=0, source_value='640999903')


語彙を渡して、同じ変換をもう一度します。
`condition_source_concept_id` が ICD-10 の concept、`condition_concept_id` がその対応先の標準の concept です。

In [29]:
cdm3 = work / "cdm_with_vocab.duckdb"
report3 = run_etl([m.yoshiki1], [m.ef], cdm3, vocab=vocab_path)

con3 = duckdb.connect(str(cdm3), read_only=True)
con3.sql("""
    SELECT condition_source_value, condition_source_concept_id, condition_concept_id,
           count(*) AS n_rows
    FROM condition_occurrence GROUP BY ALL ORDER BY condition_source_value
""").df()

,condition_source_value,condition_source_concept_id,condition_concept_id,n_rows
0,C189,900000006,900000106,18
1,E119,900000007,900000107,25
2,E785,0,0,28
3,I10,900000008,900000108,28
4,I500,900000003,900000103,17
5,I639,900000002,900000102,9
6,J189,900000001,900000101,20
7,K802,900000004,900000104,15
8,N189,0,0,28
9,N390,0,0,14


`coverage` は、concept ID が入った行数と、入ったコードの種類数を数えます。
入らなかったコードは、件数の多い順に並びます。
コマンドでは `jpomop report` で同じ表が出ます。

本物の語彙を使えば、対応づけを急ぐべきコードをこの表で確認できます。
下の割合は、このノートブックで作った架空のファイルの結果です。
実データでの割合を表すものではありません。

In [30]:
from jpomop.report import coverage

pd.DataFrame(
    [(c.table, c.rows, c.mapped_rows, f"{c.row_rate:.0%}", c.codes, c.mapped_codes, c.top_unmapped)
     for c in coverage(cdm3)],
    columns=["cdm_table", "rows", "mapped_rows", "row_rate", "codes", "mapped_codes", "unmapped_codes"],
)

,cdm_table,rows,mapped_rows,row_rate,codes,mapped_codes,unmapped_codes
0,condition_occurrence,213,143,67%,11,8,"[(E785, 28), (N189, 28), (N390, 14)]"
1,drug_exposure,692,451,65%,3,2,"[(640999903, 241)]"
2,procedure_occurrence,325,0,0%,3,0,"[(170999901, 150), (160999901, 141), (15099990..."


## 12. 表に問い合わせる例を4つ

表は DuckDB ファイルに入っているので、SQL で集計できます。
データが架空なので、下の結果自体に意味はありません。
表どうしが想定どおりにつながるかを確かめる例です。

### 主傷病ごとの入院日数

`condition_status_concept_id` が 32902 の行が主傷病です。

In [31]:
con.sql("""
    SELECT c.condition_source_value AS main_diagnosis_icd10,
           count(*) AS stays,
           round(avg(v.visit_end_date - v.visit_start_date + 1), 1) AS mean_days,
           min(v.visit_end_date - v.visit_start_date + 1) AS min_days,
           max(v.visit_end_date - v.visit_start_date + 1) AS max_days
    FROM visit_occurrence v
    JOIN condition_occurrence c USING (visit_occurrence_id)
    WHERE c.condition_status_concept_id = 32902
    GROUP BY 1 ORDER BY stays DESC
""").df()

,main_diagnosis_icd10,stays,mean_days,min_days,max_days
0,J189,20,6.5,3,10
1,C189,18,6.4,2,10
2,I500,17,5.9,2,10
3,K802,15,6.1,2,10
4,N390,14,6.1,2,10
5,S7200,11,5.5,2,10
6,I639,9,4.7,2,8


入院の数を足すと 104 で、113 になりません。
残りの 9 回は主傷病に疑いが付いているので、主傷病の行がありません。
疑い病名を外した結果が、集計にも表れています。
すべての入院を数えたい場合は、病名ではなく `visit_occurrence` から始める必要があります。

### 薬ごとの患者数と使用日数

In [32]:
con.sql("""
    SELECT d.drug_source_value,
           count(DISTINCT d.person_id) AS persons,
           count(DISTINCT d.visit_occurrence_id) AS stays,
           count(*) AS rows_,
           count(DISTINCT (d.person_id, d.drug_exposure_start_date)) AS person_days
    FROM drug_exposure d GROUP BY 1 ORDER BY 1
""").df()

,drug_source_value,persons,stays,rows_,person_days
0,620999901,92,104,243,243
1,620999902,90,97,208,208
2,640999903,93,103,241,241


### 病棟を移った入院の数

In [33]:
con.sql("""
    SELECT wards_in_stay, count(*) AS stays
    FROM (SELECT visit_occurrence_id, count(*) AS wards_in_stay
          FROM visit_detail GROUP BY 1)
    GROUP BY 1 ORDER BY 1
""").df()

,wards_in_stay,stays
0,1,88
1,2,25


### 2回以上入院した患者と、前の退院からの日数

In [34]:
con.sql("""
    SELECT person_id, visit_occurrence_id, visit_start_date, visit_end_date,
           visit_start_date - lag(visit_end_date) OVER (PARTITION BY person_id
                                                         ORDER BY visit_start_date) AS days_since_last_discharge
    FROM visit_occurrence
    QUALIFY count(*) OVER (PARTITION BY person_id) > 1
    ORDER BY person_id, visit_start_date
    LIMIT 8
""").df()

,person_id,visit_occurrence_id,visit_start_date,visit_end_date,days_since_last_discharge
0,5,5,2026-06-06,2026-06-12,<NA>
1,5,6,2026-06-18,2026-06-21,6
2,11,12,2026-06-01,2026-06-02,<NA>
3,11,13,2026-06-10,2026-06-19,8
4,12,14,2026-06-10,2026-06-17,<NA>
5,12,15,2026-06-20,2026-06-26,3
6,24,27,2026-06-03,2026-06-05,<NA>
7,24,28,2026-06-19,2026-06-24,14


## 13. Parquet で書き出す

同じ表を、表ごとに1つの Parquet ファイルとして書き出すこともできます。

In [35]:
report4 = run_etl([m.yoshiki1], [m.ef], work / "cdm_parquet", fmt="parquet")
sorted(p.name for p in (work / "cdm_parquet").iterdir())

['condition_occurrence.parquet',
 'death.parquet',
 'drug_exposure.parquet',
 'observation_period.parquet',
 'person.parquet',
 'procedure_occurrence.parquet',
 'visit_detail.parquet',
 'visit_occurrence.parquet']

## 14. このノートブックで分からないこと

- 病院の実データで jpomop がどう動くかは分かりません。実データではまだ動かしていません。
- 実際の変換率は分かりません。11 章の語彙は架空です。
- 診療行為には、まだ標準の concept が入りません。
- 点数や金額は変換していません。
- 外来のファイル、D ファイル、H ファイルは読み込みません。
- 病棟グループごとの様式1、再入院をまとめた様式1、仮の様式1 の扱いは、実施説明資料を私が読んで決めたものです。実データでの確認が要ります。

DPC データを扱っている方が、データのある環境で jpomop を動かしてくださると助かります。
データを外に出す必要はありません。件数とエラーの内容だけで十分です。
Issue は https://github.com/kojikojikojikoji/jpomop/issues で受け付けています。

下のセルで、ファイルを閉じて一時フォルダを消します。

In [36]:
con.close()
con3.close()
import shutil

shutil.rmtree(work)